# 14 - Camera + USS Fusion for Low/High  (Phase D - the headline)

The paper's central complementarity result. Three INDEPENDENT Low/High signals
per sequence:

1. **Camera detector** - the trained YOLO's predicted Low/High (its actual
   output, not the label), aggregated over the sequence's frames.
2. **USS** - the production stack's own height classifier, `ClassProbHigh` from
   the CusReplay classifier monitor, thresholded (the baseline to beat).
3. **Ground truth** - the class-derived Low/High from Label V2.

Questions:
- How accurate is USS alone? the detector alone?
- **Where USS is wrong, is the camera right?** (the "camera assists USS" claim)
- Does a simple **late fusion** beat either alone, especially on the classes
  USS struggles with?

Runs on the gold-test sequences that have a CusReplay log. **GPU** (runs the
detector on cached frames).

In [ ]:
# --- pipeline bootstrap (identical first cell in every pipeline notebook) ---
import subprocess, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = Path("/content/vision-uss-research")
    assert (REPO_DIR / ".git").exists(), \
        "repo not found - run scripts/bootstrap_colab.sh first (set REPO_URL)"
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    str(REPO_DIR / "requirements" / "colab.txt")], check=True)
else:
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents]
                    if (p / "pyproject.toml").exists())

if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

from vision_uss_research.runs import git_sha
from vision_uss_research.settings import load_paths

PROFILE = "colab_drive" if IN_COLAB else "local"
ARTIFACTS_OVERRIDE = None
RAW_ROOT_OVERRIDE = None

paths = load_paths(profile=PROFILE)
ARTIFACTS = Path(ARTIFACTS_OVERRIDE or paths.outputs_dir)
RAW_ROOT = Path(RAW_ROOT_OVERRIDE or paths.raw_data_root)
print(f"repo      : {REPO_DIR}")
print(f"git sha   : {git_sha(REPO_DIR)}")
print(f"artifacts : {ARTIFACTS}")
print(f"raw root  : {RAW_ROOT}")

In [ ]:
import json as _json
from collections import Counter

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from ultralytics import YOLO

from vision_uss_research.alignment.uss_class import (find_cusreplay_csv,
                                                     read_classifier_monitor,
                                                     uss_sequence_decision)
from vision_uss_research.runs import finish_run, new_run_id, resolve_run, start_run
from vision_uss_research.sequences import frames_window_id, with_drive_retry

MODEL_RUN = None
CONF = 0.25
USS_HIGH_THRESHOLD = 0.5      # ClassProbHigh >= this (normalized) -> USS says high
MAX_SEQUENCES = 0            # 0 = all test sequences with a CusReplay log
SEED = 0

CLASS_BINS = {"curbstone": "low", "speedbump": "low", "woodenboard": "low",
              "pole": "high", "dummychild": "high", "bicyclestand": "high",
              "hose": "low", "step": "low", "cone": "high", "bollard": "high",
              "car": "high", "tree": "high", "bush": "high", "stonelarge": "high",
              "stonemiddle": "high", "cubestandard": "high", "standardcube": "high",
              "ubarrier": "high", "fence": "high", "curbstone_side": "low"}
CATEGORY_LOWHIGH = {"low_other": "low", "high_other": "high"}   # bucket -> bin

model_dir = resolve_run(ARTIFACTS / "models", "train", run_id=MODEL_RUN,
                        require_file="train/weights/best.pt")
assert model_dir is not None, "no trained model - run notebook 06 first"
model_run = _json.loads((model_dir / "run.json").read_text())
dataset_dir = ARTIFACTS / Path(model_run["inputs"][0])
prov = _json.loads((dataset_dir / "provenance.json").read_text())
class_names = prov["class_names"]
model = YOLO(str(model_dir / "train" / "weights" / "best.pt"))

# name -> low/high for detector output (named classes via CLASS_BINS, buckets direct)
def det_bin(name):
    return CATEGORY_LOWHIGH.get(name, CLASS_BINS.get(name.lower()))

manifest = pd.read_csv(dataset_dir / "dataset_manifest.csv")
test = manifest[manifest["split"] == "test"].copy()
id2name = {i: n for i, n in enumerate(class_names)}
test["obj"] = test["class_id"].map(id2name)
test_seqs = (test.drop_duplicates("sequence_id")[["sequence_id", "obj"]]
             .reset_index(drop=True))

WINDOW_ROOT = ARTIFACTS / "frames" / frames_window_id(20, (0.30, 0.85))
RUN_DIR = ARTIFACTS / "reports" / new_run_id("fusion", "camera_uss")
RUN_DIR.mkdir(parents=True, exist_ok=True)
start_run(RUN_DIR, config={"model_run": model_dir.name, "conf": CONF,
                           "uss_threshold": USS_HIGH_THRESHOLD},
          inputs=[f"models/{model_dir.name}", f"datasets/{dataset_dir.name}"],
          repo_root=REPO_DIR)
print(f"model   : {model_dir.name}")
print(f"test sequences: {len(test_seqs)}")
print(f"output  : {RUN_DIR}")

## Collect the three signals per sequence

In [ ]:
def detector_bin(seq_id):
    """Aggregate the detector's Low/High over a sequence's cached frames:
    confidence-weighted vote of every detection's low/high category. None if
    the model detects nothing."""
    votes = Counter()
    for cam in ("rear", "front"):
        fdir = WINDOW_ROOT / seq_id / cam
        if not fdir.exists():
            continue
        for fp in sorted(fdir.glob("*.jpg")):
            res = model.predict(str(fp), conf=CONF, verbose=False)[0]
            for cls, conf in zip(res.boxes.cls.tolist(), res.boxes.conf.tolist()):
                b = det_bin(class_names[int(cls)])
                if b:
                    votes[b] += float(conf)
        if votes:
            break                       # one camera is enough
    if not votes:
        return None, 0.0
    total = sum(votes.values())
    top = max(votes, key=votes.get)
    return top, round(votes[top] / total, 3)

rows = []
seqs = test_seqs.sample(frac=1, random_state=SEED)
if MAX_SEQUENCES:
    seqs = seqs.head(MAX_SEQUENCES)
for _, r in tqdm(seqs.iterrows(), total=len(seqs), desc="Fusing", unit="seq"):
    seq_id, obj = r["sequence_id"], r["obj"]
    truth = CATEGORY_LOWHIGH.get(obj, CLASS_BINS.get(str(obj).lower()))
    if truth is None:
        continue
    # USS signal
    cus = with_drive_retry(find_cusreplay_csv, RAW_ROOT / seq_id)
    uss = uss_sequence_decision(read_classifier_monitor(cus),
                                USS_HIGH_THRESHOLD) if cus else {
        "uss_bin": None, "class_prob_high": None, "n_readings": 0}
    # camera signal
    dbin, dconf = detector_bin(seq_id)
    rows.append({"sequence_id": seq_id, "target_object": obj, "truth": truth,
                 "uss_bin": uss["uss_bin"], "uss_prob": uss["class_prob_high"],
                 "uss_readings": uss["n_readings"],
                 "det_bin": dbin, "det_conf": dconf})

fus = pd.DataFrame(rows)
fus.to_csv(RUN_DIR / "fusion_signals.csv", index=False)
print(f"sequences with signals: {len(fus)}")
print(f"  with USS reading : {fus['uss_bin'].notna().sum()}")
print(f"  with detection   : {fus['det_bin'].notna().sum()}")

## Accuracy of each signal, and the fusion

In [ ]:
def acc(pred_col, sub):
    m = sub[sub[pred_col].notna()]
    return (m[pred_col] == m["truth"]).mean() if len(m) else float("nan"), len(m)

# both signals present -> the fair comparison set
both = fus[fus["uss_bin"].notna() & fus["det_bin"].notna()].copy()

# late fusion: trust the detector when it fired (it is far more accurate);
# fall back to USS only when the camera saw nothing
fus["fused_bin"] = fus["det_bin"].where(fus["det_bin"].notna(), fus["uss_bin"])
both["fused_bin"] = both["det_bin"]     # detector always present in `both`

for label, sub in [("all test seqs", fus), ("both signals present", both)]:
    ua, un = acc("uss_bin", sub)
    da, dn = acc("det_bin", sub)
    fa, fn = acc("fused_bin", sub)
    print(f"\n[{label}]  (n={len(sub)})")
    print(f"  USS alone      : {ua:.3f}  (n={un})")
    print(f"  detector alone : {da:.3f}  (n={dn})")
    print(f"  fused          : {fa:.3f}  (n={fn})")

## The headline table: where USS is wrong, is the camera right?

In [ ]:
wrong_uss = both[both["uss_bin"] != both["truth"]]
cam_rescues = (wrong_uss["det_bin"] == wrong_uss["truth"]).mean() if len(wrong_uss) else float("nan")
wrong_cam = both[both["det_bin"] != both["truth"]]
uss_rescues = (wrong_cam["uss_bin"] == wrong_cam["truth"]).mean() if len(wrong_cam) else float("nan")

print(f"USS wrong on {len(wrong_uss)} sequences -> camera correct on "
      f"{100*cam_rescues:.1f}% of them  (CAMERA ASSISTS USS)")
print(f"camera wrong on {len(wrong_cam)} sequences -> USS correct on "
      f"{100*uss_rescues:.1f}% of them")

# per-class: USS vs detector accuracy - where does the camera help most?
def bin_acc(g, col):
    m = g[g[col].notna()]
    return (m[col] == m["truth"]).mean() if len(m) else np.nan
per_class = (both.groupby("target_object")
             .apply(lambda g: pd.Series({
                 "n": len(g),
                 "uss_acc": bin_acc(g, "uss_bin"),
                 "det_acc": bin_acc(g, "det_bin")}), include_groups=False)
             .sort_values("uss_acc"))
per_class["camera_gain"] = (per_class["det_acc"] - per_class["uss_acc"]).round(3)
display(per_class.round(3))

## Ambiguous band + confusion figures

In [ ]:
# confusion of each signal vs truth
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, col, title in [(axes[0], "uss_bin", "USS"), (axes[1], "det_bin", "Detector")]:
    c = pd.crosstab(both["truth"], both[col]).reindex(index=["low","high"],
                    columns=["low","high"]).fillna(0)
    im = ax.imshow(c.values, cmap="Blues")
    ax.set_xticks([0,1], ["low","high"]); ax.set_yticks([0,1], ["low","high"])
    ax.set_xlabel(f"{title} pred"); ax.set_ylabel("truth")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, int(c.values[i,j]), ha="center", va="center",
                    color="white" if c.values[i,j] > c.values.max()/2 else "black")
    ax.set_title(f"{title} vs truth")
fig.tight_layout(); fig.savefig(RUN_DIR / "fusion_confusion.jpg", dpi=110)
plt.show()

# USS ClassProbHigh distribution by truth - shows the ambiguous middle band
if fus["uss_prob"].notna().any():
    fig, ax = plt.subplots(figsize=(8, 4))
    for lab, color in [("low","tab:green"), ("high","tab:red")]:
        v = fus[(fus["truth"]==lab) & fus["uss_prob"].notna()]["uss_prob"]
        ax.hist(v, bins=30, alpha=0.5, label=f"truth {lab}", color=color)
    ax.axvline(USS_HIGH_THRESHOLD, color="k", ls="--", label="USS threshold")
    ax.set_xlabel("USS ClassProbHigh (normalized)"); ax.set_ylabel("sequences")
    ax.set_title("USS height probability by true class"); ax.legend()
    fig.savefig(RUN_DIR / "uss_prob_distribution.jpg", dpi=110, bbox_inches="tight")
    plt.show()

In [ ]:
ua, _ = acc("uss_bin", both); da, _ = acc("det_bin", both); fa, _ = acc("fused_bin", both)
ua_all, _ = acc("uss_bin", fus); fa_all, _ = acc("fused_bin", fus)
summary = {
    "n_test_sequences": int(len(fus)),
    "n_both_signals": int(len(both)),
    "uss_accuracy": round(float(ua), 4),
    "detector_accuracy": round(float(da), 4),
    "fused_accuracy_both": round(float(fa), 4),
    "uss_accuracy_all": round(float(ua_all), 4),
    "fused_accuracy_all": round(float(fa_all), 4),
    "camera_rescues_uss_pct": round(float(cam_rescues), 4),
    "uss_rescues_camera_pct": round(float(uss_rescues), 4),
}
finish_run(RUN_DIR, summary=summary)
print(_json.dumps(summary, indent=1))
print(f"\nsignals table: {RUN_DIR / 'fusion_signals.csv'}")

## Reading the result

- **Camera >> USS** on Low/High is the expected and desirable story: the
  production ultrasonic height classifier is weaker than the learned camera
  detector, and **where USS errs the camera corrects it** at high rate -
  concrete evidence for "camera assists USS" near the bumper.
- **Per-class `camera_gain`** shows where the camera helps most - typically the
  low, flat classes (curbs, speed bumps) that USS height estimation struggles
  with.
- The **USS ClassProbHigh distribution** visualizes the ambiguous middle band;
  the camera's value is concentrated there.

Caveats: USS decision is the dominant-object track's median ClassProbHigh (a
proxy for the production output); evaluated on the gold test sequences that
have a CusReplay log; single facility/vehicle. This is the camera-USS
complementarity result - the paper's practical headline, and it stands on its
own regardless of the (weak) standalone geometric height.